# Process to be used by frontend code

Initial iteration to plot clusters on UI from Google Colab outputs

In [35]:
import ast
import json
import re
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction import text as sklearn_text

In [36]:
ARTWORK_WEIGHT = 0.25  # must match the value used in text_extraction.ipynb
SHARED_THRESHOLD = 0.70  # primary cluster weight below this → artist is "shared"


def _parse_is_artwork(val):
    if isinstance(val, bool):
        return val
    if isinstance(val, str):
        return val.strip().lower() in {"true", "1", "yes"}
    return bool(val)


def _build_featured_quote(points):
    def _is_artwork(point):
        value = point.get("is_artwork")
        if isinstance(value, str):
            return value.strip().lower() in {"true", "1", "yes"}
        return bool(value)

    candidate_points = [point for point in points if not _is_artwork(point)]
    if not candidate_points:
        return None

    def _sort_key(point):
        distance = point.get("distance_to_center")
        distance_value = float(distance) if distance is not None else np.inf
        source_idx = str(point.get("source_idx") or "")
        point_value = point.get("point")
        point_sort_value = point_value if point_value is not None else np.inf
        return (distance_value, source_idx, point_sort_value)

    selected = min(candidate_points, key=_sort_key)
    return {
        "source_idx": selected.get("source_idx"),
        "point": selected.get("point"),
        "text": selected.get("text"),
    }


def _semantic_grid_order(center_vectors, occupied):
    """
    Greedy nearest-neighbour traversal of cluster centers in PCA space.
    Returns a list of cluster IDs in visually semantic grid order.
    """
    ids = sorted(occupied)
    if len(ids) <= 1:
        return ids

    coords = {}
    for c in ids:
        vec = center_vectors[c]
        # simple 2-element PCA projection inline — no extra import needed
        coords[c] = vec

    # use the mean inter-cluster distance vector as a 1D sort key via dot with first PC
    mat = np.array([coords[c] for c in ids])
    mat = mat - mat.mean(axis=0)
    _, _, vt = np.linalg.svd(mat, full_matrices=False)
    pc1 = vt[0]
    scores = {c: float(np.dot(coords[c], pc1)) for c in ids}

    # greedy nearest-neighbour starting from the lowest PC1 score
    remaining = sorted(ids, key=lambda c: scores[c])
    order = [remaining.pop(0)]
    while remaining:
        last = order[-1]
        nearest = min(remaining, key=lambda c: np.linalg.norm(coords[c] - coords[last]))
        order.append(nearest)
        remaining.remove(nearest)
    return order


def process_cluster_file(filepath):
    with open(filepath) as f:
        data = json.load(f)

    num_clusters_out = []

    for version in data:
        k = version["n"]
        k_clusters = len(version["clusters"])

        rows = []
        for cluster in version["clusters"]:
            for point in cluster["data"]:
                row = {
                    "cluster": int(cluster["key"]),
                    "keywords": cluster["keywords"],
                }
                row.update(point)
                rows.append(row)

        df = pd.DataFrame(rows)
        df["_weight"] = df["is_artwork"].apply(_parse_is_artwork).apply(
            lambda x: ARTWORK_WEIGHT if x else 1.0
        )

        keyword_map = {int(c["key"]): c["keywords"] for c in version["clusters"]}

        center_vectors = np.array(
            [
                [
                    df[df["cluster"] == c][f"dist_to_cluster_{cc}"].mean()
                    for cc in range(k_clusters)
                ]
                for c in range(k_clusters)
            ]
        )

        # --- per-artist rows ---
        occupied_clusters = set()
        artist_rows = []

        for artist_id, group in df.groupby("artist_id"):
            cluster_weights = group.groupby("cluster")["_weight"].sum()
            primary_cluster = int(cluster_weights.idxmax())
            occupied_clusters.add(primary_cluster)

            total_weight = group["_weight"].sum()
            cluster_distribution = {
                str(c): round(float(group[group["cluster"] == c]["_weight"].sum()) / total_weight, 4)
                for c in range(k_clusters)
            }

            avg_dists = {}
            for c in range(k_clusters):
                vals = group[f"dist_to_cluster_{c}"].values
                weights = group["_weight"].values
                if weights.sum() == 0 or np.isnan(vals).all():
                    avg_dists[c] = None
                else:
                    mean_val = np.average(vals, weights=weights)
                    avg_dists[c] = None if np.isnan(mean_val) else round(float(mean_val), 4)

            primary_weight = float(cluster_distribution.get(str(primary_cluster), 0))
            is_shared = primary_weight < SHARED_THRESHOLD

            artist_rows.append({
                "artistId": artist_id,
                "primaryGroup": primary_cluster,
                "isShared": is_shared,
                "clusterDistribution": cluster_distribution,
                "_avg_dists": avg_dists,
            })

        # --- semantic grid order for occupied clusters ---
        grid_order = _semantic_grid_order(center_vectors, occupied_clusters)
        grid_index_map = {c: i for i, c in enumerate(grid_order)}

        # --- build groups list ---
        groups = []
        for c in grid_order:
            groups.append({
                "id": c,
                "gridIndex": grid_index_map[c],
                "keywords": keyword_map.get(c, []),
                "featured_quote": _build_featured_quote(
                    [p for p in rows if p["cluster"] == c]
                ),
            })

        # --- assign ring layout to exclusive artists ---
        exclusive_by_cluster = defaultdict(list)
        for row in artist_rows:
            if not row["isShared"]:
                exclusive_by_cluster[row["primaryGroup"]].append(row)

        for c, cluster_rows in exclusive_by_cluster.items():
            dist_key = c
            sorted_rows = sorted(cluster_rows, key=lambda r: r["_avg_dists"].get(dist_key) or 0.5)
            n_items = len(sorted_rows)
            layer_count = min(5, max(2, int(np.ceil(np.sqrt(n_items)))))
            per_layer = int(np.ceil(n_items / layer_count))
            r_min_frac, r_max_frac = 0.16, 1.0

            layer_items = defaultdict(list)
            for layer in range(layer_count):
                for row in sorted_rows[layer * per_layer:(layer + 1) * per_layer]:
                    layer_items[layer].append(row)

            phase = grid_index_map.get(c, 0) * (np.pi / max(n_items, 6))
            for layer_index, items in layer_items.items():
                m = len(items)
                frac = layer_index / max(layer_count - 1, 1)
                radius = r_min_frac + frac * (r_max_frac - r_min_frac)
                for j, row in enumerate(items):
                    row["_angle"] = round(float((j / max(m, 1)) * 2 * np.pi + phase), 4)
                    row["_radius_frac"] = round(float(radius), 4)

        # --- assign jitter angles to shared artists ---
        shared_rows = [r for r in artist_rows if r["isShared"]]
        for j, row in enumerate(shared_rows):
            row["_jitter_angle"] = round((j / max(len(shared_rows), 1)) * 2 * np.pi, 4)

        # --- build final artist list ---
        artists_out = []
        for row in artist_rows:
            entry = {
                "artistId": row["artistId"],
                "primaryGroup": row["primaryGroup"],
                "gridIndex": grid_index_map.get(row["primaryGroup"], 0),
                "isShared": row["isShared"],
                "clusterDistribution": row["clusterDistribution"],
            }
            if row["isShared"]:
                entry["_jitter_angle"] = row.get("_jitter_angle", 0.0)
                entry["_radius_frac"] = 1.0
            else:
                entry["_angle"] = row.get("_angle", 0.0)
                entry["_radius_frac"] = row.get("_radius_frac", 0.5)
            artists_out.append(entry)

        num_clusters_out.append({
            "k": k,
            "n": len(groups),
            "groups": groups,
            "artists": artists_out,
        })

    return num_clusters_out

In [37]:
def add_exhibition_ring_layout(items, r_inner=0.35, r_outer=0.85):
    sorted_items = sorted(items, key=lambda r: (r["isShared"], r["artistId"]))

    layer_items = defaultdict(list)
    for row in sorted_items:
        layer = 1 if row["isShared"] else 0
        layer_items[layer].append(row)

    for layer_index, layer_rows in layer_items.items():
        m = len(layer_rows)
        phase = layer_index * (np.pi / max(m, 6))
        radius = r_outer if layer_index == 1 else r_inner
        for j, row in enumerate(layer_rows):
            row["_angle"] = round(float((j / max(m, 1)) * 2 * np.pi + phase), 4)
            row["_radius_frac"] = round(float(radius), 4)

    return sorted_items

In [38]:
def write_json(path, data):
    with open(path, "w") as f:
        json.dump(data, f, indent=2, allow_nan=False)

## Artist/institution cluster outputs

In [39]:
artist_num_clusters = process_cluster_file("../../data/colab/ar_clusters.json")
institution_num_clusters = process_cluster_file("../../data/colab/inst_clusters.json")

In [40]:
write_json("../../data/clusters/artist_cluster_summary.json", {"numClusters": artist_num_clusters})
write_json("../../data/clusters/institution_cluster_summary.json", {"numClusters": institution_num_clusters})

## Exhibition layout output

In [41]:
df = pd.read_csv("../../data/colab/exhibitions_with_labels.csv")
df["artist_ids"] = df["artists"].apply(ast.literal_eval)

all_artist_occurrences = Counter()
for ids in df["artist_ids"]:
    for aid in ids:
        all_artist_occurrences[aid] += 1

R_INNER = 0.35
R_OUTER = 0.85

exhibitions_out = []

for _, row in df.iterrows():
    artist_ids = row["artist_ids"]
    labels = ast.literal_eval(row["labels"]) if isinstance(row["labels"], str) else row["labels"]

    items = [
        {
            "artistId": aid,
            "isShared": all_artist_occurrences[aid] > 1,
        }
        for aid in artist_ids
    ]

    items = add_exhibition_ring_layout(items, r_inner=R_INNER, r_outer=R_OUTER)

    exhibitions_out.append(
        {
            "id": int(row["id"]),
            "name": row["name"],
            "labels": labels[:3],
            "items": items,
        }
    )

write_json("../../data/clusters/exhibition_clusters.json", exhibitions_out)

In [42]:
# new exhibitions format dedupe artist points
df = pd.read_csv("../../data/colab/exhibitions_with_labels.csv")
df["artist_ids"] = df["artists"].apply(ast.literal_eval)

def parse_labels(x):
    if isinstance(x, str):
        try: return ast.literal_eval(x)
        except: return []
    return x or []

df["labels_parsed"] = df["labels"].apply(parse_labels)

artist_exhibitions = {}
for _, row in df.iterrows():
    for aid in row["artist_ids"]:
        artist_exhibitions.setdefault(aid, [])
        artist_exhibitions[aid].append(int(row["id"]))

R_INNER = 0.35
R_OUTER = 0.85

exclusive_by_exhibition = {}
for aid, eids in artist_exhibitions.items():
    if len(eids) == 1:
        exclusive_by_exhibition.setdefault(eids[0], []).append(aid)

exclusive_angle_map = {}
for eid, aids in exclusive_by_exhibition.items():
    for j, aid in enumerate(sorted(aids)):
        exclusive_angle_map[aid] = round((j / max(len(aids), 1)) * 2 * np.pi, 4)

shared_artists = [aid for aid, eids in artist_exhibitions.items() if len(eids) > 1]
for j, aid in enumerate(shared_artists):
    pass 

exhibitions_out = []
for _, row in df.iterrows():
    exhibitions_out.append({
        "id": int(row["id"]),
        "name": row["name"],
        "labels": row["labels_parsed"][:3],
    })

artists_out = []
for j, (aid, eids) in enumerate(artist_exhibitions.items()):
    is_shared = len(eids) > 1
    entry = {
        "artistId": aid,
        "exhibitionIds": eids,
        "isShared": is_shared,
        "_radius_frac": R_OUTER if is_shared else R_INNER,
    }
    if not is_shared:
        entry["_angle"] = exclusive_angle_map.get(aid, 0.0)
    else:
        entry["_jitter_angle"] = round((j / max(len(shared_artists), 1)) * 2 * np.pi, 4)
    artists_out.append(entry)

output_object = {"exhibitions": exhibitions_out, "artists": artists_out}

write_json("../../data/clusters/exhibition_clusters.json", output_object)